# Tutorial 11: DepMap Cancer Dependency Analysis

This notebook demonstrates how to load and analyse **DepMap** (Cancer
Dependency Map) data using `embpy`. The [DepMap](https://depmap.org/)
project provides genome-wide CRISPR knockout screening data across
hundreds of cancer cell lines, enabling identification of **essential
genes** — those whose loss selectively kills cancer cells.

We focus on **CRISPRGeneEffect** (CERES scores), where negative values
indicate a gene is essential for cell survival.

### What you'll learn

1. Load DepMap CRISPR data as an AnnData via `embpy.pp.load_depmap`
2. Explore cell-line metadata and filter by cancer lineage
3. Identify pan-essential and lineage-specific essential genes
4. Compute gene embeddings and correlate with dependency profiles
5. Visualise dependency landscapes with PCA

### Requirements

Download the DepMap 24Q4 data first (see `data/depmap_data/download_depmap.py`).

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import embpy

DATA_DIR = "/lustre/groups/ml01/workspace/goncalo.pinto/embpy/data/datasets/depmap_data"

## 1. Available DepMap datasets

See what datasets are registered in the loader.

In [ ]:
print("Registered DepMap datasets:")
for name in embpy.pp.list_depmap_datasets():
    card = embpy.pp.depmap_info(name)
    print(f"  {name:50s}  [{card.data_type}]")
    print(f"    {card.description[:80]}...")

## 2. Load CRISPRGeneEffect

This loads the CERES gene-effect scores into an AnnData object.
Cell-line metadata from `Model.csv` is automatically joined into `.obs`.

In [ ]:
adata = embpy.pp.load_depmap("CRISPRGeneEffect", data_dir=DATA_DIR)
adata

In [ ]:
print(f"Cell lines (obs): {adata.n_obs}")
print(f"Genes (var):      {adata.n_vars}")
print(f"\nObs columns (metadata): {list(adata.obs.columns[:15])}")
print(f"\nVar columns: {list(adata.var.columns)}")
print(f"\nFirst 5 genes:")
adata.var.head()

## 3. Explore cell-line metadata

DepMap covers cancer cell lines across many tissue lineages.

In [ ]:
lineage_counts = adata.obs["OncotreeLineage"].value_counts()
print(f"Number of lineages: {len(lineage_counts)}\n")

fig, ax = plt.subplots(figsize=(12, 6))
lineage_counts.head(20).plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Number of cell lines")
ax.set_title("Top 20 cancer lineages in DepMap")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Distribution of gene-effect scores

CERES scores near 0 mean the gene has no effect on fitness when knocked
out. Negative scores indicate the gene is essential (cell death upon
knockout). A score of -1 corresponds to the median effect of known
common essential genes.

In [ ]:
mean_effects = np.nanmean(adata.X, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(mean_effects, bins=100, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].axvline(-0.5, color="red", linestyle="--", label="Common essential threshold")
axes[0].set_xlabel("Mean CERES score across cell lines")
axes[0].set_ylabel("Number of genes")
axes[0].set_title("Distribution of mean gene-effect scores")
axes[0].legend()

std_effects = np.nanstd(adata.X, axis=0)
axes[1].scatter(mean_effects, std_effects, s=1, alpha=0.3, color="steelblue")
axes[1].set_xlabel("Mean CERES score")
axes[1].set_ylabel("Std CERES score")
axes[1].set_title("Mean vs. variability of gene effects")
axes[1].axvline(-0.5, color="red", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Pan-essential genes

Genes that are essential across most cell lines (low mean CERES score)
are likely core fitness genes (e.g. ribosomal proteins, proteasome
subunits, spliceosome components).

In [ ]:
gene_stats = pd.DataFrame({
    "gene_symbol": adata.var["gene_symbol"].values,
    "mean_effect": mean_effects,
    "std_effect": std_effects,
    "frac_dependent": np.nanmean(adata.X < -0.5, axis=0),
}, index=adata.var.index)

pan_essential = gene_stats.sort_values("mean_effect").head(30)
print("Top 30 pan-essential genes (most negative mean CERES):")
pan_essential[["gene_symbol", "mean_effect", "frac_dependent"]]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(
    pan_essential["gene_symbol"].values[::-1],
    pan_essential["mean_effect"].values[::-1],
    color="crimson",
    edgecolor="white",
    linewidth=0.3,
)
ax.set_xlabel("Mean CERES score")
ax.set_title("Top 30 pan-essential genes")
ax.axvline(-1.0, color="black", linestyle="--", alpha=0.3, label="Median common essential")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Lineage-specific dependencies

Some genes are essential only in specific cancer types. Let's compare
two lineages to find **selective dependencies**.

In [ ]:
lineage_a = "Lung"
lineage_b = "Breast"

mask_a = adata.obs["OncotreeLineage"] == lineage_a
mask_b = adata.obs["OncotreeLineage"] == lineage_b

mean_a = np.nanmean(adata.X[mask_a.values], axis=0)
mean_b = np.nanmean(adata.X[mask_b.values], axis=0)
diff = mean_a - mean_b

gene_symbols = adata.var["gene_symbol"].values

print(f"{lineage_a}: {mask_a.sum()} cell lines")
print(f"{lineage_b}: {mask_b.sum()} cell lines")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(mean_a, mean_b, s=2, alpha=0.3, color="grey")

# Highlight top selective genes for each lineage
n_label = 10
top_a_idx = np.argsort(diff)[:n_label]   # more essential in A than B
top_b_idx = np.argsort(diff)[-n_label:]  # more essential in B than A

ax.scatter(mean_a[top_a_idx], mean_b[top_a_idx], s=30, color="tab:blue", zorder=3)
ax.scatter(mean_a[top_b_idx], mean_b[top_b_idx], s=30, color="tab:red", zorder=3)

for idx in top_a_idx:
    ax.annotate(gene_symbols[idx], (mean_a[idx], mean_b[idx]),
                fontsize=7, color="tab:blue", ha="left")
for idx in top_b_idx:
    ax.annotate(gene_symbols[idx], (mean_a[idx], mean_b[idx]),
                fontsize=7, color="tab:red", ha="left")

lims = [min(mean_a.min(), mean_b.min()) - 0.1, 0.5]
ax.plot(lims, lims, "k--", alpha=0.3)
ax.set_xlabel(f"Mean CERES — {lineage_a}")
ax.set_ylabel(f"Mean CERES — {lineage_b}")
ax.set_title(f"Selective dependencies: {lineage_a} vs {lineage_b}")
plt.tight_layout()
plt.show()

## 7. Load expression data

We can also load RNA-seq expression to correlate with CRISPR effects.

In [ ]:
adata_expr = embpy.pp.load_depmap(
    "OmicsExpressionProteinCodingGenesTPMLogp1",
    data_dir=DATA_DIR,
)
print(adata_expr)
print(f"\nExpression matrix: {adata_expr.n_obs} cell lines × {adata_expr.n_vars} genes")

## 8. Gene-effect PCA across cell lines

PCA on the gene-effect matrix reveals cell-line clusters that share
similar dependency profiles — often grouping by tissue of origin.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

X_imputed = SimpleImputer(strategy="mean").fit_transform(adata.X)

pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_imputed)

print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")

In [ ]:
top_lineages = lineage_counts.head(8).index.tolist()

fig, ax = plt.subplots(figsize=(12, 9))
colors = plt.cm.tab10(np.linspace(0, 1, len(top_lineages)))

# Plot "Other" first in grey
other_mask = ~adata.obs["OncotreeLineage"].isin(top_lineages)
ax.scatter(
    pca_coords[other_mask.values, 0],
    pca_coords[other_mask.values, 1],
    s=8, alpha=0.2, color="lightgrey", label="Other",
)

for i, lineage in enumerate(top_lineages):
    mask = (adata.obs["OncotreeLineage"] == lineage).values
    ax.scatter(
        pca_coords[mask, 0],
        pca_coords[mask, 1],
        s=15, alpha=0.7, color=colors[i], label=lineage, edgecolors="k", linewidths=0.2,
    )

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA of CRISPR gene-effect profiles")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, markerscale=2)
plt.tight_layout()
plt.show()

## 9. Correlate dependency with gene embeddings

We can use `embpy`'s PPI embeddings to ask: **do genes with similar
PPI embeddings have correlated dependency profiles?**

This tests whether functional proximity in the PPI network predicts
co-essentiality across cancer cell lines.

In [ ]:
from embpy.models.ppi_models import PrecomputedPPIWrapper
from numpy.linalg import norm

PPI_DATA_DIR = "/lustre/groups/ml01/workspace/goncalo.pinto/embpy/data/embeddings/precomputed_embeddings_string"

import torch
ppi = PrecomputedPPIWrapper(
    data_dir=PPI_DATA_DIR,
    species=9606,
    embedding_type="functional",
)
ppi.load(torch.device("cpu"))

print(f"PPI genes available: {len(ppi.available_genes)}")

In [ ]:
depmap_genes = set(adata.var["gene_symbol"].values)
ppi_genes = set(ppi.available_genes)
shared_genes = sorted(depmap_genes & ppi_genes)
print(f"Genes in both DepMap and PPI: {len(shared_genes)}")

# Subset to shared genes
sample_genes = shared_genes[:500]

# Get PPI embeddings
ppi_embs = np.array(ppi.embed_batch(sample_genes))

# Get dependency profiles (genes × cell lines)
gene_sym_to_var_idx = {sym: i for i, sym in enumerate(adata.var["gene_symbol"].values)}
dep_indices = [gene_sym_to_var_idx[g] for g in sample_genes]
dep_profiles = adata.X[:, dep_indices].T  # (n_genes, n_cell_lines)
dep_profiles = np.nan_to_num(dep_profiles, nan=0.0)

print(f"PPI embedding matrix: {ppi_embs.shape}")
print(f"Dependency profile matrix: {dep_profiles.shape}")

In [ ]:
from scipy.spatial.distance import pdist, squareform

# Cosine similarity matrices
ppi_sim = 1 - squareform(pdist(ppi_embs, metric="cosine"))
dep_sim = 1 - squareform(pdist(dep_profiles, metric="cosine"))

# Correlation between PPI similarity and co-essentiality
upper_tri = np.triu_indices(len(sample_genes), k=1)
ppi_flat = ppi_sim[upper_tri]
dep_flat = dep_sim[upper_tri]

from scipy.stats import spearmanr
rho, pval = spearmanr(ppi_flat, dep_flat)
print(f"Spearman correlation between PPI similarity and co-essentiality: ρ={rho:.4f}, p={pval:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Subsample for plotting
n_plot = min(50000, len(ppi_flat))
idx = np.random.choice(len(ppi_flat), n_plot, replace=False)

ax.scatter(ppi_flat[idx], dep_flat[idx], s=1, alpha=0.1, color="steelblue")
ax.set_xlabel("PPI embedding cosine similarity")
ax.set_ylabel("Co-essentiality (dependency cosine similarity)")
ax.set_title(f"PPI similarity vs co-essentiality (Spearman ρ={rho:.3f})")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 10. Store in AnnData for downstream analysis

We can store the dependency data alongside gene embeddings in a single
AnnData for use with `embpy`'s benchmark pipeline.

In [ ]:
adata.obsm["X_pca_crispr"] = pca_coords

print(adata)
print(f"\nStored keys:")
print(f"  .obs:  {list(adata.obs.columns[:10])} ...")
print(f"  .var:  {list(adata.var.columns)}")
print(f"  .obsm: {list(adata.obsm.keys())}")
print(f"  .uns:  {list(adata.uns.keys())}")

## Summary

| Step | Code |
|---|---|
| List datasets | `embpy.pp.list_depmap_datasets()` |
| Dataset info | `embpy.pp.depmap_info("CRISPRGeneEffect")` |
| Load as AnnData | `embpy.pp.load_depmap("CRISPRGeneEffect", data_dir=...)` |
| Load expression | `embpy.pp.load_depmap("OmicsExpressionProteinCodingGenesTPMLogp1", ...)` |
| Gene symbols | `adata.var["gene_symbol"]` |
| Cell-line metadata | `adata.obs["OncotreeLineage"]`, `adata.obs["CellLineName"]` |
| Essential genes | `mean_effect < -0.5` |